# 第2日 ノートブック02：条件分岐・繰り返し・関数

# Day 2 Notebook 02: Conditions, Loops and Functions

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. `if` / `elif` / `else` で条件分岐を書ける | 1. Write `if` / `elif` / `else` branches |
| 2. `for` ループで複数のデータを処理できる | 2. Process many values with a `for` loop |
| 3. リストを作り、要素を取り出せる | 3. Create lists and access their elements |
| 4. 引数と戻り値のある関数を書ける | 4. Write functions with parameters and return values |
| 5. 例外処理で異常な入力を扱える | 5. Handle invalid input with exceptions |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 100 分 / 100 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第2日ノートブック01 を終えていること。 / You have finished Day 2 Notebook 01. |
| 必要なライブラリ / Required libraries | numpy, matplotlib, pillow |
| 使用カーネル / Jupyter kernel | aivision |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 測定値判定プログラム（複数測定対応） | A measurement judgement program (multiple measurements) |
| 入力チェック付きの関数 | A validated function |

## 本日の学習タイムライン / Learning timeline for today

| 時刻 / Time | 内容（日本語） | Activity (English) |
| --- | --- | --- |
| 08:00–09:00 | 講師：if と関数の説明 | Instructor: `if` and functions |
| 09:00–10:30 | Part 1〜Part 3（if・ループ・リスト） | Part 1-Part 3 (if, loops, lists) |
| 10:30–10:40 | 休憩 | Short break |
| 10:40–12:00 | Part 4〜Part 6（関数・例外・テスト） | Part 4-Part 6 (functions, exceptions, tests) |
| 12:00–12:50 | 昼休み | Lunch break |
| 12:50–13:50 | 講師：振り返りと午後の説明 | Instructor: review and briefing |
| 13:50–16:50 | 演習・提出物 | Exercises and submission |

> **日本語：** 12:00–12:50 は昼休みです。08:00–09:00 と 12:50–13:50 は講師の時間、それ以外は自分のペースで進めてください。
>
> **English:** 12:00-12:50 is the lunch break. 08:00-09:00 and 12:50-13:50 are instructor sessions; the remaining time is self-paced work.

---

## Part 0　準備 / Preparation

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    """研修フォルダ（AI_Vision_Training）を上方向に探す。
    Walk upwards until the course root folder is found."""
    current = Path(start).resolve() if start else Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "assets").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
        "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
    )

COURSE_ROOT = find_course_root()
ASSETS = COURSE_ROOT / "assets"
print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

In [ ]:
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("numpy", "numpy"),
    ("matplotlib", "matplotlib"),
    ("PIL", "pillow"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
import platform
from pathlib import Path

from matplotlib import font_manager, rcParams

def find_japanese_font():
    """日本語を表示できるフォントを探す。見つからなければ None を返す。
    Locate a font that can render Japanese. Returns None when nothing is found."""
    system = platform.system()
    if system == "Windows":
        folders = [Path("C:/Windows/Fonts")]
    elif system == "Darwin":
        folders = [Path("/System/Library/Fonts"), Path("/Library/Fonts"), Path.home() / "Library/Fonts"]
    else:
        folders = [Path("/usr/share/fonts"), Path("/usr/local/share/fonts"), Path.home() / ".fonts"]

    preferred = [
        "NotoSansJP-Regular", "NotoSansCJK-Regular", "YuGothR", "meiryo", "msgothic",
        "BIZ-UDGothicR", "NotoSansJP", "HiraginoSans",
    ]
    found = []
    for folder in folders:
        if not folder.is_dir():
            continue
        for path in folder.rglob("*"):
            if path.suffix.lower() not in {".ttf", ".ttc", ".otf"}:
                continue
            name = path.stem
            if any(key.lower() in name.lower() for key in preferred):
                found.append(path)

    def rank(path):
        name = path.stem.lower()
        return (0 if "regular" in name or name.endswith("r") else 1, len(name))

    return sorted(found, key=rank)[0] if found else None


japanese_font = find_japanese_font()
if japanese_font is None:
    print("日本語フォントが見つかりません / no Japanese font found.")
    print("図のタイトルが □ になる場合は、英字のタイトルを使ってください。")
    print("If figure titles show boxes, use English titles instead.")
else:
    font_manager.fontManager.addfont(str(japanese_font))
    family_name = font_manager.FontProperties(fname=str(japanese_font)).get_name()
    rcParams["font.family"] = [family_name, "DejaVu Sans"]
    rcParams["axes.unicode_minus"] = False
    print("図の日本語フォント / figure font:", family_name)
    print("  ->", japanese_font)

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

def show_png(relative_parts, caption=""):
    """素材フォルダの PNG を表示する / display a PNG from the assets folder."""
    path = ASSETS.joinpath(*relative_parts)
    if not path.is_file():
        print("画像がありません / file not found:", path)
        return
    plt.figure(figsize=(8, 5))
    plt.imshow(Image.open(path))
    plt.axis("off")
    if caption:
        plt.title(caption)
    plt.show()

---

## Part 1　条件分岐 / Conditions

#### 日本語：`if` で「もし〜なら」を書く
**日本語：** `if` は「もし条件が正しければ、この処理をする」という意味です。

```python
if 条件:
    処理
```

**必ず必要なもの**

1. 条件のあとの **`:`（コロン）**
2. 次の行の **インデント（4 文字の空白）**

この 2 つを忘れると `SyntaxError` になります。

**English:** `if` means "if the condition is true, do this".

```python
if condition:
    do_something
```

Two things are mandatory:

1. a **colon `:`** after the condition
2. **indentation** (4 spaces) on the next line

Forgetting either produces a `SyntaxError`.
#### English: Writing 'if' with `if`

In [ ]:
show_png(["diagrams", "diagram_control_flow.png"], "if / else chooses one path")

In [ ]:
measured_mm = 100.3
nominal_mm = 100.0
tolerance_mm = 0.5

difference = measured_mm - nominal_mm

if abs(difference) <= tolerance_mm:
    print("判定 / result : OK")
else:
    print("判定 / result : NG")

print("差 / difference:", round(difference, 3), "mm")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
判定 / result : OK
差 / difference: 0.3 mm
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
result     : OK
difference : 0.3 mm
```

#### 日本語：`elif` で 3 つ以上に分ける
**日本語：** 3 つ以上に分けたいときは `elif`（else if）を使います。
**上から順に** 判定され、**最初に当てはまった処理だけ** が実行されます。

```python
if 条件1:
    処理1
elif 条件2:
    処理2
else:
    処理3
```

**日本語：** 順番が大切です。`if x < 10` を先に書くと、`x = 5` は
`elif x < 20` まで到達しません。

**English:** Use `elif` (else if) for three or more branches. Conditions are tested **top to
bottom**, and **only the first match runs**.

```python
if condition1:
    action1
elif condition2:
    action2
else:
    action3
```

Order matters: if `x < 10` comes first, then `x = 5` never reaches `elif x < 20`.
#### English: Splitting into three or more with `elif`

In [ ]:
def grade_width(difference_mm, tolerance_mm=0.5):
    """差から 4 段階の判定を返す / return a four-level judgement."""
    size = abs(difference_mm)
    if size <= tolerance_mm * 0.4:
        return "A (very good / 非常に良い)"
    elif size <= tolerance_mm:
        return "B (OK / 規格内)"
    elif size <= tolerance_mm * 2:
        return "C (warning / 注意)"
    else:
        return "D (reject / 不良)"


for value in [-2.0, -0.6, -0.15, 0.0, 0.15, 0.6, 2.0]:
    print(f"差 {value:6.2f} mm -> {grade_width(value)}")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
差  -2.00 mm -> D (reject / 不良)
差  -0.60 mm -> C (warning / 注意)
差  -0.15 mm -> A (very good / 非常に良い)
差   0.00 mm -> A (very good / 非常に良い)
差   0.15 mm -> A (very good / 非常に良い)
差   0.60 mm -> C (warning / 注意)
差   2.00 mm -> D (reject / 不良)
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
diff  -2.00 mm -> D (reject)
diff  -0.60 mm -> C (warning)
diff  -0.15 mm -> A (very good)
diff   0.00 mm -> A (very good)
diff   0.15 mm -> A (very good)
diff   0.60 mm -> C (warning)
diff   2.00 mm -> D (reject)
```

---

## Part 2　リストと繰り返し / Lists and loops

#### 日本語：リストは「値の並び」
**日本語：** リストは、複数の値を 1 つにまとめたものです。

```python
measurements = [100.0, 100.3, 99.7]
```

| 操作 | 例 | 結果 |
|---|---|---|
| 個数 | `len(measurements)` | `3` |
| 取り出す | `measurements[0]` | `100.0` |
| 最後 | `measurements[-1]` | `99.7` |
| 範囲 | `measurements[0:2]` | `[100.0, 100.3]` |
| 追加 | `measurements.append(100.1)` | 末尾に追加 |
| 合計 | `sum(measurements)` | 合計値 |
| 最大 | `max(measurements)` | 最大値 |

> **日本語：** 番号は 0 からです。`measurements[1]` は **2 番目** です。

**English:** A list groups several values into one.

| Operation | Example | Result |
|---|---|---|
| length | `len(measurements)` | `3` |
| element | `measurements[0]` | `100.0` |
| last | `measurements[-1]` | `99.7` |
| slice | `measurements[0:2]` | `[100.0, 100.3]` |
| append | `measurements.append(100.1)` | adds at the end |
| sum | `sum(measurements)` | total |
| max | `max(measurements)` | largest |

Indexing starts at 0, so `measurements[1]` is the **second** element.
#### English: A list is a sequence of values

In [ ]:
measurements_mm = [100.0, 100.3, 99.7, 100.6, 99.4]

print("個数 / count      :", len(measurements_mm))
print("1 番目 / first    :", measurements_mm[0])
print("最後 / last       :", measurements_mm[-1])
print("2 番目 / second   :", measurements_mm[1])
print("平均 / mean       :", round(sum(measurements_mm) / len(measurements_mm), 3))
print("最大 / max        :", max(measurements_mm))
print("最小 / min        :", min(measurements_mm))

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
個数 / count      : 5
1 番目 / first    : 100.0
最後 / last       : 99.4
2 番目 / second   : 100.3
平均 / mean       : 100.0
最大 / max        : 100.6
最小 / min        : 99.4
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
count    : 5
first    : 100.0
last     : 99.4
second   : 100.3
mean     : 100.0
max      : 100.6
min      : 99.4
```

#### 日本語：`for` で全要素を処理する
**日本語：** `for` は「リストの中身を 1 つずつ取り出して、同じ処理をする」という意味です。

```python
for value in measurements:
    print(value)
```

- `value` は毎回作り直される変数です（名前は自由）
- リストの要素数だけ繰り返します
- **インデントされた部分** が繰り返されます

**日本語：** 「100 個の測定値を 1 つずつ判定する」——これが自動化の基本形です。

**English:** `for` takes each element of a list and runs the same code on it.

```python
for value in measurements:
    print(value)
```

- `value` is a new variable on every pass (any name works)
- the loop runs once per element
- only the **indented block** repeats

Judging 100 measurements one by one — this is the basic shape of automation.
#### English: Processing every element with `for`

In [ ]:
def judge(measured_mm, nominal_mm=100.0, tolerance_mm=0.5):
    """規格内なら "OK"、そうでなければ "NG" を返す / return OK or NG."""
    difference = measured_mm - nominal_mm
    if abs(difference) <= tolerance_mm:
        return "OK"
    return "NG"


ng_count = 0
for index, value in enumerate(measurements_mm):
    result = judge(value)
    if result == "NG":
        ng_count = ng_count + 1
    print(f"{index}: {value:6.1f} mm -> {result}")

print()
print("NG の件数 / NG count:", ng_count)

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
0:  100.0 mm -> OK
1:  100.3 mm -> OK
2:   99.7 mm -> OK
3:  100.6 mm -> NG
4:   99.4 mm -> NG

NG の件数 / NG count: 2
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
0:  100.0 mm -> OK
1:  100.3 mm -> OK
2:   99.7 mm -> OK
3:  100.6 mm -> NG
4:   99.4 mm -> NG

NG count: 2
```

---

## Part 3　関数 / Functions

#### 日本語：関数は「名前を付けた処理」
**日本語：** 関数は、何度も使う処理に名前を付けたものです。

```python
def 関数名(引数):
    処理
    return 戻り値
```

| 言葉 | 意味 | 例 |
|---|---|---|
| `def` | 関数を定義する | `def judge(...)` |
| 引数（ひきすう） | 呼び出すときに渡す値 | `judge(100.3)` の `100.3` |
| 戻り値 | 関数が返す結果 | `"OK"` |
| `return` | 値を返して関数を終了する | `return "OK"` |

**関数を使う 3 つの利点**

1. **同じコードを書かなくて済む**
2. **名前で意味が分かる**（`judge(100.3)` は一目で分かる）
3. **修正が 1 か所で済む**（ルールが変わったら関数の中だけ直す）

**English:** A function gives a name to a block of processing you use repeatedly.

```python
def function_name(parameters):
    processing
    return value
```

| Term | Meaning | Example |
|---|---|---|
| `def` | defines a function | `def judge(...)` |
| parameter | a value you pass in | the `100.3` in `judge(100.3)` |
| return value | what the function gives back | `"OK"` |
| `return` | sends the value back and ends the function | `return "OK"` |

Three benefits: no duplicated code, a meaningful name, and one place to change when the rule changes.
#### English: A function is a named block of processing

In [ ]:
def judge_width(measured_mm, nominal_mm=100.0, tolerance_mm=0.5):
    """幅を判定する / judge a width.

    Parameters
    ----------
    measured_mm : float
        測定値 / the measured value
    nominal_mm : float
        規格の中心値 / the nominal value
    tolerance_mm : float
        許容差 / the allowed deviation

    Returns
    -------
    str
        "OK" または "NG"
    """
    difference = measured_mm - nominal_mm
    if abs(difference) <= tolerance_mm:
        return "OK"
    return "NG"


print(judge_width(100.3))
print(judge_width(100.6))
print(judge_width(97.0))
print(judge_width(97.0, tolerance_mm=3.0))

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
OK
NG
NG
OK
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
OK
NG
NG
OK
```

---

## Part 4　例外処理 / Exception handling

#### 日本語：`try` / `except` で異常を扱う
**日本語：** 現場では、必ずしも正しい値が来るとは限りません。
空欄、文字、`None`、極端に大きな値……。

`try` / `except` を使うと、**エラーで止めずに** 対応できます。

```python
try:
    危険な処理
except ValueError:
    失敗したときの処理
```

**日本語：** 大切なのは、**エラーを隠さない** ことです。
「とりあえず `except: pass`」は絶対にやめてください。
**何が起きたかを記録する** のが正しい使い方です。

**English:** Real data is not always clean: blanks, letters, `None`, absurd values.

`try` / `except` lets you **handle the failure instead of crashing**.

```python
try:
    risky_operation
except ValueError:
    what_to_do_when_it_fails
```

Crucially, **do not hide errors**. Never write a bare `except: pass`. Always **record what
happened**.
#### English: Handling abnormal input with `try` / `except`

In [ ]:
raw_values = ["100.1", "99.8", "", "not-a-number", "100.4", " 100.0 "]

good_values = []
rejected = []

for raw in raw_values:
    try:
        value = float(raw)
    except (TypeError, ValueError) as error:
        rejected.append((raw, str(error)))
        continue
    good_values.append(value)

print("採用した値 / accepted :", good_values)
print("却下した値 / rejected :")
for raw, reason in rejected:
    print(f"   {raw!r:>16} -> {reason}")
print()
print("再現率の例 / acceptance rate:", f"{len(good_values) / len(raw_values) * 100:.1f} %")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
採用した値 / accepted : [100.1, 99.8, 100.4, 100.0]
却下した値 / rejected :
              '' -> could not convert string to float: ''
    'not-a-number' -> could not convert string to float: 'not-a-number'

再現率の例 / acceptance rate: 66.7 %
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
accepted : [100.1, 99.8, 100.4, 100.0]
rejected :
              '' -> could not convert string to float: ''
    'not-a-number' -> could not convert string to float: 'not-a-number'

acceptance rate: 66.7 %
```

#### 日本語：入力チェック付きの関数
**日本語：** 実務では、関数の入口で値をチェックします。
これを **入力チェック（バリデーション）** と呼びます。

チェックする内容の例：

- 数値か
- マイナスでないか
- 現実的な範囲か（シート幅が 5,000 mm を超えたら測定ミス）

**English:** In practice, validate values at the entrance of a function. This is called
**input validation**.

Things to check:

- is it a number
- is it negative
- is it plausible (a 5,000 mm sheet width means a measurement error)
#### English: A function with validation

In [ ]:
def judge_width_checked(measured_mm, nominal_mm=100.0, tolerance_mm=0.5,
                        minimum_mm=0.0, maximum_mm=5000.0):
    """入力チェック付きの判定 / judgement with input validation.

    Raises
    ------
    TypeError
        数値でない場合 / when the value is not a number
    ValueError
        現実的でない値の場合 / when the value is not plausible
    """
    for name, value in [("measured_mm", measured_mm), ("nominal_mm", nominal_mm),
                        ("tolerance_mm", tolerance_mm)]:
        if not isinstance(value, (int, float)) or isinstance(value, bool):
            raise TypeError(f"{name} must be a number, got {type(value).__name__}")
    if tolerance_mm <= 0:
        raise ValueError(f"tolerance_mm must be positive, got {tolerance_mm}")
    if not (minimum_mm <= measured_mm <= maximum_mm):
        raise ValueError(f"measured_mm={measured_mm} is outside {minimum_mm}..{maximum_mm}")

    return "OK" if abs(measured_mm - nominal_mm) <= tolerance_mm else "NG"


test_cases = [
    ("normal / 正常", 100.3),
    ("out of spec / 規格外", 100.8),
    ("text / 文字列", "100.3"),
    ("absurd / ありえない値", 99999.0),
    ("negative tolerance / 負の許容差", None),
]

for label, value in test_cases:
    try:
        if value is None:
            result = judge_width_checked(100.0, tolerance_mm=-1.0)
        else:
            result = judge_width_checked(value)
        print(f"{label:<28} -> {result}")
    except (TypeError, ValueError) as error:
        print(f"{label:<28} -> {type(error).__name__}: {error}")

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
normal / 正常                -> OK
out of spec / 規格外         -> NG
text / 文字列                -> TypeError: measured_mm must be a number, got str
absurd / ありえない値        -> ValueError: measured_mm=99999.0 is outside 0.0..5000.0
negative tolerance / 負の許容差 -> ValueError: tolerance_mm must be positive, got -1.0
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
normal / OK ...
TypeError / ValueError as described
```

---

## Part 5　簡単なテスト / Simple testing

#### 日本語：`assert` で自動チェックする
**日本語：** `assert` は「この条件が正しいはず」と宣言する文です。
正しければ何も起きず、間違っていればエラーになります。

```python
assert judge_width(100.0) == "OK"
```

**日本語：** プログラムを書き換えたとき、`assert` が全部通れば
「前と同じ動作をしている」と確認できます。これを **回帰テスト** と呼びます。

**English:** `assert` states "this condition should hold". Nothing happens when it holds; an error
appears when it does not.

```python
assert judge_width(100.0) == "OK"
```

After changing your code, if all `assert`s still pass you know the behaviour is unchanged.
That is a **regression test**.
#### English: Automatic checks with `assert`

In [ ]:
def run_tests():
    """判定関数のテスト / tests for the judgement function."""
    assert judge_width(100.0) == "OK", "nominal should be OK"
    assert judge_width(100.4) == "OK", "just inside tolerance"
    assert judge_width(100.6) == "NG", "just outside tolerance"
    assert judge_width(99.6) == "OK", "just inside, negative side"
    assert judge_width(99.4) == "NG", "just outside, negative side"
    assert judge_width(100.5) == "OK", "exactly at tolerance"
    assert judge_width(97.0, tolerance_mm=3.0) == "OK", "wider tolerance"
    print("すべてのテストが通りました / all tests passed")


run_tests()

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
すべてのテストが通りました / all tests passed
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
all tests passed
```

#### 日本語：テストが失敗するとどうなるか
**日本語：** 次のセルは、**わざと間違った期待値** を書いています。
`AssertionError` が出ることを確認してください。これは正常な動作です。

**English:** The next cell deliberately asserts something **false**. You should see an
`AssertionError`. That is the expected behaviour.
#### English: What a failing test looks like

In [ ]:
try:
    assert judge_width(100.6) == "OK", "100.6 mm should this really be OK?"
    print("テスト成功 / test passed (unexpected)")
except AssertionError as error:
    print("テスト失敗 / test failed:", error)
    print("-> 期待値または関数のどちらかが間違っています / either the expectation or the function is wrong")

---

## Part 6　測定値判定プログラムを完成させる / Completing the measurement program

#### 日本語：要求仕様
**日本語：** 次の仕様でプログラムを作ります。

| 項目 | 値 |
|---|---|
| 規格中心 | 100.0 mm |
| 許容差 | ±0.5 mm |
| 入力 | 測定値のリスト（文字列を含む可能性あり） |
| 出力 | 各測定値の判定、OK/NG 件数、NG 率、最悪値 |

**English:** Build a program with these requirements.

| Item | Value |
|---|---|
| nominal | 100.0 mm |
| tolerance | ±0.5 mm |
| input | a list of measurements (may contain text) |
| output | per-item judgement, OK/NG count, NG rate, worst case |
#### English: Requirements

In [ ]:
def analyse_measurements(raw_values, nominal_mm=100.0, tolerance_mm=0.5):
    """測定値リストをまとめて判定する / judge a whole list of measurements.

    Returns
    -------
    dict
        判定結果のまとめ / a summary dictionary
    """
    accepted = []
    rejected = []
    results = []

    for raw in raw_values:
        try:
            value = float(raw)
        except (TypeError, ValueError):
            rejected.append(raw)
            continue
        accepted.append(value)
        results.append({"value": value, "result": judge_width(value, nominal_mm, tolerance_mm)})

    ng_values = [item["value"] for item in results if item["result"] == "NG"]
    worst = max(ng_values, key=lambda v: abs(v - nominal_mm)) if ng_values else None

    return {
        "total_input": len(raw_values),
        "accepted": accepted,
        "rejected": rejected,
        "results": results,
        "ok_count": len(results) - len(ng_values),
        "ng_count": len(ng_values),
        "ng_rate_percent": round(len(ng_values) / len(results) * 100, 2) if results else 0.0,
        "worst_value": worst,
    }


data = ["100.0", "100.3", "", "99.7", "100.6", "99.4", "abc", "100.5"]
summary = analyse_measurements(data)

print("入力件数 / inputs      :", summary["total_input"])
print("採用 / accepted        :", len(summary["accepted"]), summary["accepted"])
print("却下 / rejected        :", summary["rejected"])
print("OK / NG                :", summary["ok_count"], "/", summary["ng_count"])
print("NG 率 / NG rate        :", summary["ng_rate_percent"], "%")
print("最悪値 / worst value   :", summary["worst_value"])

### 実行結果の例 / Expected output

**日本語：** 次のような結果になります（数値は環境によって多少変わります）。

```text
入力件数 / inputs      : 8
採用 / accepted        : 6 [100.0, 100.3, 99.7, 100.6, 99.4, 100.5]
却下 / rejected        : ['', 'abc']
OK / NG                : 4 / 2
NG 率 / NG rate        : 33.33 %
最悪値 / worst value   : 100.6
```

**English:** You should see something like the following (exact numbers vary slightly between machines).

```text
inputs      : 8
accepted    : 6
rejected    : ['', 'abc']
OK / NG     : 4 / 2
NG rate     : 33.33 %
worst value : 100.6
```

---

## 演習 / Exercises

#### 日本語：演習 1（基本）：厚さの判定に変える
**日本語：** 上の `analyse_measurements` を、**厚さ 2.00 ± 0.10 mm** の判定に使ってください。
`nominal_mm=2.0`, `tolerance_mm=0.1` を指定して、次のデータを判定してください。

`["2.00", "2.05", "2.11", "1.95", "2.20"]`

**English:** Use `analyse_measurements` for a thickness specification of 2.00 ± 0.10 mm by passing
`nominal_mm=2.0, tolerance_mm=0.1`, and judge `["2.00", "2.05", "2.11", "1.95", "2.20"]`.
#### English: Exercise 1 (basic): switch to thickness
Done when you can state which two values are NG and why (2.11 and 2.20).

In [ ]:
# 演習 1 / Exercise 1
thickness_data = ["2.00", "2.05", "2.11", "1.95", "2.20"]
thickness_summary = analyse_measurements(thickness_data, nominal_mm=2.0, tolerance_mm=0.1)
for item in thickness_summary["results"]:
    print(f'{item["value"]:5.2f} -> {item["result"]}')
print("NG 件数 / NG count:", thickness_summary["ng_count"])

#### 日本語：演習 2（中級）：複数規格に対応する
**日本語：** 幅・厚さ・長さの 3 つの規格をまとめて判定できるようにしてください。
規格を辞書（`dict`）で定義し、`for` で回してください。

| 項目 | 規格 | 許容差 |
|---|---|---|
| width | 100.0 | 0.5 |
| thickness | 2.00 | 0.10 |
| length | 2500.0 | 2.0 |

**English:** Judge three specifications (width, thickness, length) together. Define them in a
dictionary and loop over it.
#### English: Exercise 2 (intermediate): support several specifications
Done when a single loop prints one result line per specification.

In [ ]:
# 演習 2 / Exercise 2
specifications = {
    "width":     {"nominal": 100.0,  "tolerance": 0.5,  "value": 100.3},
    "thickness": {"nominal": 2.00,   "tolerance": 0.10, "value": 2.11},
    "length":    {"nominal": 2500.0, "tolerance": 2.0,  "value": 2502.5},
}

for name, spec in specifications.items():
    # judge_width を使って判定してください / use judge_width
    result = "?"
    print(f"{name:<10} {spec['value']:>10} mm -> {result}")

#### 日本語：演習 3（発展）：異常値の記録を残す
**日本語：** `analyse_measurements` を改良して、**却下した値とその理由** を
`rejected` にタプル `(値, 理由)` で残すようにしてください。
最後に却下理由を集計し、理由ごとの件数を表示してください。

**English:** Improve `analyse_measurements` so that `rejected` contains `(value, reason)` tuples.
Then count how many times each reason occurred and print the counts.
#### English: Exercise 3 (challenge): keep a record of anomalies
Done when the printout shows each rejection reason with its count.

In [ ]:
# 演習 3 / Exercise 3
# ヒント / hint:  except (TypeError, ValueError) as error: rejected.append((raw, str(error)))

# 理由の集計例 / counting example
from collections import Counter

example_reasons = ["could not convert string to float: ''", "could not convert string to float: 'abc'"]
print(Counter(example_reasons))

---

#### 日本語：理解度チェック / Understanding questions
**日本語：**

1. `if` に `:` を忘れるとどうなりますか。
2. `elif` を 3 つ並べたとき、2 つ目と 3 つ目の条件が両方成り立つ場合はどうなりますか。
3. 関数を作る利点を 2 つ挙げてください。
4. `for` ループの中で `continue` を使うと、何が起きますか（自分で調べてください）。
5. 「とりあえず `except: pass`」が危険なのはなぜですか。

**English:**

1. What happens if you forget the `:` after `if`?
2. With three `elif` branches, what happens when the second and third conditions are both true?
3. Give two benefits of writing a function.
4. What happens when you use `continue` inside a `for` loop? (Look it up.)
5. Why is a bare `except: pass` dangerous?
#### English: Understanding questions
Model answers: `docs/instructor/day02_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

**日本語：** エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

**English:** When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `IndentationError: expected an indented block` | `IndentationError: expected an indented block` | `if` の次の行を 4 文字下げてください。 | Indent the line after `if` by four spaces. |
| `NameError: name 'judge_width' is not defined` | `NameError: name 'judge_width' is not defined` | 関数を定義したセルを実行していません。上から順に実行し直してください。 | The cell defining the function was not run. Run the cells from the top. |
| `AssertionError: just outside tolerance` | `AssertionError: just outside tolerance` | テストの期待値か関数のどちらかが間違っています。両方を print して確認してください。 | Either the expected value or the function is wrong. Print both. |
| `ValueError` が関数の中で出る | A `ValueError` raised inside the function | 入力チェックが働いています。メッセージにどの値が問題か書かれています。 | Your validation worked; the message names the offending value. |
| ループが最後の要素を処理しない | The loop skips the last element | `range(len(...))` を使うと起こりがちです。`for value in items:` を使ってください。 | Common with `range(len(...))`. Prefer `for value in items:`. |

## 完了チェックリスト / Completion checklist

**日本語：** すべてにチェックが付いたら、このノートブックは完了です。

**English:** This notebook is complete when every box is checked.

- [ ] Part 1〜Part 6 を読み、すべてのコードセルを実行した
      - [ ] Read Parts 1-6 and ran every code cell
- [ ] 演習 1〜3 を実施した
      - [ ] Completed exercises 1-3
- [ ] `analyse_measurements` が動くことを確認した
      - [ ] Confirmed `analyse_measurements` works
- [ ] `assert` によるテストを書いた
      - [ ] Wrote `assert` based tests
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Python 制御フロー](https://docs.python.org/ja/3/tutorial/controlflow.html)
- [Python エラーと例外](https://docs.python.org/ja/3/tutorial/errors.html)
- [Python 組み込み関数](https://docs.python.org/ja/3/library/functions.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

**日本語：** `for` と `while` の違いを調べ、「終了条件が分からない繰り返し」はどちらで書くべきか考えてください。

**English:** Look up the difference between `for` and `while` and decide which to use when the number of repetitions is unknown.